# معمل التجربة والتطبيق العملي: المسترجع الهجين في LangChain
## Playground: Combining Dense and Sparse Retriever with LangChain (`Lecture 42`)

مرحباً بك في المعمل العملي للمحاضرة 42 من مسار **RAG Mastery**.

---

### أهداف المعمل:
1. تجربة بناء مسترجع كثيف ومسترجع متناثر بشكل مستقل.
2. استخدام `EnsembleRetriever` لدمج النتائج واختبار تأثير تغيير الأوزان (`weights`).
3. تجربة استعلامات دلالية (`Semantic Paraphrase`) واستعلامات حرفية (`Exact Acronym / Code`).
4. ربط المسترجع الهجين بسلسلة RAG كاملة للحصول على إجابة نهائية.


In [1]:
# 1. إعداد البيئة واستيراد المكتبات
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers.bm25 import BM25Retriever

try:
    from langchain.retrievers.ensemble import EnsembleRetriever
except ImportError:
    from langchain_classic.retrievers.ensemble import EnsembleRetriever

try:
    from langchain.chains.combine_documents import create_stuff_documents_chain
    from langchain.chains import create_retrieval_chain
except ImportError:
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain
    from langchain_classic.chains import create_retrieval_chain

from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

print("[INFO] المكتبات جاهزة للعمل!")


C:\Users\aliashraf\AppData\Local\Temp\ipykernel_19644\85148797.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


[INFO] المكتبات جاهزة للعمل!


In [ ]:
# 2. تجهيز المستندات وإنشاء المسترجعين (الكثيف والمتناثر)
docs = [
    Document(page_content="LangChain helps build LLM applications", metadata={"source": "doc1"}),
    Document(page_content="LangChain can be used to develop Agentic AI applications", metadata={"source": "doc2"}),
    Document(page_content="LangChain has many types of retrievers", metadata={"source": "doc3"}),
    Document(page_content="Pinecone is a vector database for semantic search", metadata={"source": "doc4"}),
    Document(page_content="The Eiffel Tower is located in Paris and is a famous tourist attraction", metadata={"source": "doc5"})
]


In [7]:
# Dense Retriever
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


# Sparse Retriever
sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k = 3

# hybrid retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7, 0.3]  # Adjust weights as needed 0.7 for dense retriever and 0.3 for sparse retriever
) 


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:

# Experiment 1: Semantic Query
query_semantic = "tools to construct intelligent reasoning systems"

# Experiment 2: Lexical Query
query_lexical = "retrievers"


def compare_retrievers(query):
    print(f"\n{'='*60}")
    print(f"QUERY: {query}")
    print(f"{'='*60}")

    retrievers = {
        "Dense Retriever": dense_retriever,
        "Sparse Retriever": sparse_retriever,
        "Hybrid Retriever": hybrid_retriever
    }

    for name, retriever in retrievers.items():
        results = retriever.invoke(query)

        print(f"\n[{name}]")

        for i, doc in enumerate(results, 1):
            print(
                f"Result {i}: {doc.page_content} "
                f"(Source: {doc.metadata['source']})"
            )


# Run experiments
compare_retrievers(query_semantic)
compare_retrievers(query_lexical)



QUERY: tools to construct intelligent reasoning systems

[Dense Retriever]
Result 1: LangChain can be used to develop Agentic AI applications (Source: doc2)
Result 2: Pinecone is a vector database for semantic search (Source: doc4)
Result 3: LangChain helps build LLM applications (Source: doc1)

[Sparse Retriever]
Result 1: LangChain can be used to develop Agentic AI applications (Source: doc2)
Result 2: The Eiffel Tower is located in Paris and is a famous tourist attraction (Source: doc5)
Result 3: Pinecone is a vector database for semantic search (Source: doc4)

[Hybrid Retriever]
Result 1: LangChain can be used to develop Agentic AI applications (Source: doc2)
Result 2: Pinecone is a vector database for semantic search (Source: doc4)
Result 3: LangChain helps build LLM applications (Source: doc1)
Result 4: The Eiffel Tower is located in Paris and is a famous tourist attraction (Source: doc5)

QUERY: retrievers

[Dense Retriever]
Result 1: LangChain has many types of retrievers (Sou

In [14]:
# 1. Create Prompt Template
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template("""
You are a helpful assistant that provides information based on the retrieved documents.

Context:
{context}

Question: {input}

Answer:
""")

# 2. Initialize LLM
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.7,
    max_tokens=200
)

# 3. Create Stuff Documents Chain
documents_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt_template
)

# 4. Create Full RAG Chain
rag_chain = create_retrieval_chain(
    retriever=hybrid_retriever,
    combine_docs_chain=documents_chain
)

# 5. Test RAG
response = rag_chain.invoke({
    "input": "What is LangChain?"
})

print(response["answer"])

LangChain is a framework that helps developers build applications powered by large language models (LLMs). It provides a variety of tools—including many types of retrievers for semantic search—and supports building both standard LLM‑based apps and more advanced agentic AI systems.


In [15]:
print("\nRetrieved Context:")

for i, doc in enumerate(response["context"], 1):
    print(f"{i}. {doc.page_content}")
    print(f"Source: {doc.metadata['source']}")
    


Retrieved Context:
1. LangChain has many types of retrievers
Source: doc3
2. LangChain helps build LLM applications
Source: doc1
3. LangChain can be used to develop Agentic AI applications
Source: doc2
4. The Eiffel Tower is located in Paris and is a famous tourist attraction
Source: doc5
5. Pinecone is a vector database for semantic search
Source: doc4
